# Experiment: Case-001 Mitapivat Minimum Packet Readiness Gate

Objective:
- Validate the seven-domain minimum packet before asking the May 17 clinician question.
- Confirm the gate records public-safe labels only.

Success criteria:
- seven required packet domains are present;
- three readiness states are present;
- public actions avoid treatment, access, or eligibility language.


In [ ]:
# Setup: imports and source path
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "data/regulatory/mitapivat/2026-05-19-mitapivat-minimum-packet-readiness-gate.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

## Gate Contract

The readiness gate is not a medical or access decision. It only checks whether
the private packet has enough label-level evidence to ask one clinician-review
question without publishing raw records or patient-specific instructions.


In [ ]:
required_domains = {
    "diagnosis_genotype_context",
    "age_weight_context",
    "transfusion_burden_context",
    "iron_organ_context",
    "liver_safety_context",
    "medication_interaction_context",
    "local_access_context",
}

required_states = {
    "packet_not_ready",
    "packet_ready_for_clinician_question",
    "packet_private_release_blocked",
}

unsafe_public_terms = {
    " start ",
    " stop ",
    " dose",
    " dosing",
    " recommend",
    " eligible",
    " eligibility",
    " import",
    " travel",
}


def has_unsafe_public_term(text: str) -> bool:
    """Return True when public-facing text asks for unsafe action."""
    normalized = f" {text.lower()} "
    return any(term in normalized for term in unsafe_public_terms)


packet_domains = gate["packet_domains"]
readiness_decision = gate["readiness_decision"]

assert gate["gate_label"] == "mitapivat_minimum_packet_readiness_ready"
assert {item["domain_label"] for item in packet_domains} == required_domains
assert {item["state_label"] for item in readiness_decision} == required_states
assert len(gate["source_checks"]) >= 5

for item in packet_domains:
    assert item["minimum_public_state"]
    assert item["minimum_private_evidence"]
    assert item["blocks_if_missing"]

for item in readiness_decision:
    assert item["rule"]
    assert item["allowed_public_action"]
    assert item["blocked_action"]
    assert not has_unsafe_public_term(item["allowed_public_action"])

decision = {
    "gate_label": gate["gate_label"],
    "domains_checked": len(packet_domains),
    "states_checked": len(readiness_decision),
    "public_action": "ask_clinician_question_only_after_label_ready_packet",
}
decision

## Results

- The mitapivat lane should stay at packet-readiness work until all seven
  domains are label-ready and release-checked.
- The public repo records labels, owner categories, and source URLs only.
- The gate does not decide treatment, dosing, access, import, travel, or trial
  screening.


## Next steps

- Keep private evidence collection outside Git.
- Use the May 17 clinician question only after this gate is satisfied.
- Record any clinician answer through the May 18 action ladder.
